In [73]:

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt 
import time

n_s = 300
d = torch.load(f'./data/2layer_interpol_ns{n_s}_tagged.pt', weights_only=True)
d_in = 2

stau   = d["stau"].double()    
kratio = d["kratio"].double()
zdis   = d["zdis"].double()
tag    = d["tag"].double()
W = d["tag"].double()

In [74]:
if ('use_existing' not in locals()) or (use_existing != 1):
    N = 5000 # Number of training points
    epsilon = 0 # Will add with noise 
    nL = 4 # Number of hidden layer
    ns = 300
    node = 32 # Number of nodes in each hidden layer
    epochs = 100000# number of epochs in training
    lr = 0.01; eps = 1e-12;
    tolp= 10 
    plot_interval = 5000
    out_file= f'./model/classifier_ns{ns}_p{tolp}.pt' # output data file name
else:
    print('parameters: N = %d, nL = %d, node = %d, epochs = %d'%(N, nL, node, epochs))

In [75]:
class Net(nn.Module):
    def __init__(self,nL,node):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(d_in,node)
        self.fc2 = nn.Linear(node,node)
        if nL > 2:
            self.fc3 = nn.Linear(node,node)
        if nL > 3:
            self.fc4 = nn.Linear(node,node)
        if nL > 4:
            self.fc5 = nn.Linear(node,node)
        if nL > 5:
            self.fc6 = nn.Linear(node,node)
        if nL > 6:
            self.fc7 = nn.Linear(node,node)
        if nL > 7:
            self.fc8 = nn.Linear(node,node)
        self.fcn = nn.Linear(node,1)
    def forward(self, x):
        x = F.sigmoid(self.fc1(x)) #x = F.relu(self.fc1(x))
        x = F.sigmoid(self.fc2(x))
        if nL > 2:
            x = F.sigmoid(self.fc3(x))
        if nL > 3:
            x = F.sigmoid(self.fc4(x))
        if nL > 4:
            x = F.sigmoid(self.fc5(x))
        if nL > 5:
            x = F.sigmoid(self.fc6(x))
        if nL > 6:
            x = F.sigmoid(self.fc7(x))
        if nL > 7:
            x = F.sigmoid(self.fc8(x))
        return self.fcn(x)

In [79]:
def train():
    start = time.time()
    model = Net(nL, node).to(device).double()
    model.train()
    tol = 0.5 * 0.1**tolp

    optimizer = optim.Adam(model.parameters(), lr=lr, eps=eps, weight_decay=0.0)
    criterion = nn.BCEWithLogitsLoss()
    decision = []

    for epoch in range(epochs):
        optimizer.zero_grad()
        output = model(xtrain)
        loss = criterion(output, ytrain)
        loss.backward()
        optimizer.step()

        if epoch == 0:
            print(f'initial loss = {loss.item():.6e}, tol = {tol:.1e}')

        if (epoch + 1) % 1000 == 0:
            print(f'Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4e}')
            print('CA time = %s' % (time.ctime(time.time() - 3600*7)))

            with torch.no_grad():
                x_min, x_max = xtrain[:, 0].min() - 0.5, xtrain[:, 0].max() + 0.5
                y_min, y_max = xtrain[:, 1].min() - 0.5, xtrain[:, 1].max() + 0.5
                xx, yy = np.meshgrid(
                    np.linspace(x_min.item(), x_max.item(), 100),
                    np.linspace(y_min.item(), y_max.item(), 100))

                #third = xtrain[:, 2].mean().item()   # hold feature 3 fixed
                grid = np.c_[xx.ravel(), yy.ravel(), np.full(xx.size, third)]
                grid = torch.tensor(grid, dtype=torch.float64, device=device)

                preds = model(grid)
                preds = (preds > 0).long().reshape(xx.shape).cpu().numpy()

                decision.append((xx, yy, preds))

        if loss.item() < tol:
            print(f'converged at epoch {epoch}')
            break

    end = time.time()
    totaltime = end - start
    CAtime = time.ctime(end - 3600*7)
    print('Elapsed time = %.2fs, CA time = %s' % (totaltime, CAtime))

    return model, decision

In [80]:
def save_model_vars():
    # save model and key variables to a data file
    # torch.save({
    #         'epoch': epoch,
    #         'model_state_dict': model.state_dict(),
    #         'optimizer_state_dict': optimizer.state_dict(),
    #         'loss': loss
    #         },'d_m_training.pt')
    # model_scripted = torch.jit.script(model)
    # model_scripted.save('d_m_scripted.pt')
    torch.save({
            'd': d_in,'N': N,
            'nL': nL, 'node': node, 'epoch': epochs, 'tolp': tolp,
            'lr': lr, 'eps': eps,
            'xtrain': xtrain, 'ytrain': ytrain,
            'dict': model.state_dict()
            }, out_file)

In [81]:
if ('func_def_only' not in locals()) or (func_def_only != 1):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    train_data = torch.stack([stau[:,0], stau[:,1]], dim=-1)
    xtrain = train_data.to(device=device, dtype=torch.float64)
    ytrain = tag.to(device=device, dtype=torch.float64).reshape(-1, 1)
    model, decision = train()
    save_model_vars()

initial loss = 7.294247e-01, tol = 5.0e-11
Epoch 1000/100000, Loss: 4.2458e-03
CA time = Fri Jul 24 12:18:04 2026


NameError: name 'third' is not defined

In [ ]:

newtag = model(xtrain)

import os
os.makedirs('./data', exist_ok=True)
W = W.reshape(-1,1)
print("\n[6] SAVING TRAINING DATA (5 keys)")
# Create dictionary with 4 keys (all on CUDA)
training_dict = {
    'stau': stau,
    'W': W,
    'kratio': kratio,
    'zdis': zdis,
    'newtag': tag
}

# Save to .pt file
dat_file = f'./data/2layer_interpol_ns{n_s}_classified.pt'
torch.save(training_dict, dat_file)

In [ ]:
W.shape